In [ ]:
#Imports
import pandas as pd 
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import re
pd.options.display.float_format = '{:20,.2f}'.format
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
pd.set_option('display.max_columns', None)
import plotly.express as px
from folium.plugins import HeatMap
import folium
import numpy as np 
from folium.plugins import HeatMapWithTime
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
from rasterio.mask import mask
import requests
from datetime import datetime, timedelta
from tqdm import tqdm

## Extraccion y creacion Galicia shapefile GDF

In [ ]:
## Extraccion Galicia municipalities shapefile 
import geopandas as gpd
import rasterio

# Load Galicia municipalities shapefile (adjust path)
gdf_galicia = gpd.read_file('path/to/galicia_municipalities.gpkg')

# Check CRS of shapefile and reproject later if needed
print("Shapefile CRS:", gdf_galicia.crs)

# Preview first rows
gdf_galicia.head()


## Extraccion datos NASA POWER

In [ ]:

# NASA POWER parameters
nasa_params = ",".join([
    "ALLSKY_SFC_SW_DWN",  # Surface solar radiation (W/m^2)
    "T2MWET",             # Wet bulb temperature (°C)
    "PS",                 # Surface pressure (kPa)
    "QV2M"                # Specific humidity (g/kg)
])

# Province coordinates
province_coords = {
    "A_Coruña": {"lat": 43.3623, "lon": -8.4115},
    "Lugo": {"lat": 43.0097, "lon": -7.5560},
    "Ourense": {"lat": 42.3360, "lon": -7.8648},
    "Pontevedra": {"lat": 42.4334, "lon": -8.6475}
}

# Date range
start_date = datetime(2000, 1, 1)
end_date = datetime(2025, 7, 13)
chunk_size = 5 * 365  # 5 years per chunk

# Function to call NASA API for one province and chunk range
def fetch_nasa_chunk(province, lat, lon, start, end):
    start_fmt = start.strftime("%Y%m%d")
    end_fmt = end.strftime("%Y%m%d")

    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?start={start_fmt}&end={end_fmt}"
        f"&latitude={lat}&longitude={lon}"
        f"&parameters={nasa_params}"
        f"&community=AG"
        f"&format=JSON"
    )

    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Failed chunk for {province}: {start_fmt} to {end_fmt}")
        return None

    json_data = response.json()
    if "properties" not in json_data or "parameter" not in json_data["properties"]:
        print(f"❌ Invalid structure for {province}: {start_fmt} to {end_fmt}")
        return None

    param_data = json_data["properties"]["parameter"]
    dfs = []
    for param, values in param_data.items():
        df = pd.DataFrame.from_dict(values, orient="index", columns=[param])
        dfs.append(df)

    df_chunk = pd.concat(dfs, axis=1)
    df_chunk.index = pd.to_datetime(df_chunk.index)
    return df_chunk

# Fetch all data for one province
def fetch_province_nasa(province, lat, lon):
    print(f"🌞 Fetching NASA data for {province}...")
    all_chunks = []
    current_start = start_date

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=chunk_size), end_date)
        df_chunk = fetch_nasa_chunk(province, lat, lon, current_start, current_end)
        if df_chunk is not None:
            all_chunks.append(df_chunk)
        current_start = current_end + timedelta(days=1)

    if all_chunks:
        df_prov = pd.concat(all_chunks).sort_index()
        # Rename columns
        df_prov.columns = [f"{col}_{province}" for col in df_prov.columns]
        return df_prov
    else:
        print(f"⚠️ No data collected for {province}")
        return None

# Loop through provinces and combine
dfs_nasa = []
for prov, coords in province_coords.items():
    df = fetch_province_nasa(prov, coords["lat"], coords["lon"])
    if df is not None:
        dfs_nasa.append(df)

# Merge all into a single wide-format DataFrame
if dfs_nasa:
    df_nasa_all = pd.concat(dfs_nasa, axis=1).sort_index()
    df_nasa_all = df_nasa_all.loc["2000-01-01":]  # just in case
    df_nasa_all.to_csv("nasa_weather_by_province.csv")
    print("✅ NASA weather data saved to 'nasa_weather_by_province.csv'")
    display(df_nasa_all.head())
else:
    print("❌ No NASA dataframes available.")


## Extraccion Raster Data de GHS POBLACION

In [ ]:
import geopandas as gpd
import pandas as pd
import rasterio
from rasterstats import zonal_stats
import os
from tqdm import tqdm
import numpy as np
from shapely import wkt

# -----------------------------
# 1. Load Galicia Municipalities GeoDataFrame
# -----------------------------
csv_path = "/workspaces/JMT1ST-BOOSTING-ALG/data/Cleaned_and_processed_data/gdf_galicia_clean.csv"
df = pd.read_csv(csv_path)
df["geometry"] = df["geometry"].apply(wkt.loads)  # Convert WKT to shapely geometry
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")  # Now it's a GeoDataFrame

# -----------------------------
# 2. Precompute municipality areas in km²
# -----------------------------
gdf = gdf.to_crs("EPSG:25829")  # UTM for accurate area
gdf["area_km2"] = gdf.geometry.area / 1e6
gdf = gdf.to_crs("EPSG:4326")  # Return to WGS84

# -----------------------------
# 3. Load GHS-POP Raster Data
# -----------------------------
pop_dir = "/workspaces/JMT1ST-BOOSTING-ALG/data/raw/GHS DATA/GHS-POPULATION"
years = list(range(2000, 2026, 5))  # [2000, 2005, ..., 2025]
pop_data = []

print("⏳ Extracting population from rasters...")
for year in tqdm(years):
    tif_path = [f for f in os.listdir(pop_dir) if str(year) in f and f.endswith(".tif")]
    if not tif_path:
        continue
    tif_file = os.path.join(pop_dir, tif_path[0])

    # Compute population sum per municipality
    stats = zonal_stats(
        gdf.to_crs("EPSG:4326"),
        tif_file,
        stats=["sum"],
        nodata=-9999
    )

    year_data = pd.DataFrame({
        "municipality_code": gdf["municipality_code"],
        "year": year,
        "POP": [x["sum"] for x in stats]
    })

    pop_data.append(year_data)

# -----------------------------
# 4. Interpolate Missing Years (Fix: ensure POP is numeric)
# -----------------------------
df_pop = pd.concat(pop_data)

# Convert to numeric to fix interpolation issue
df_pop["POP"] = pd.to_numeric(df_pop["POP"], errors="coerce")

# Interpolate across years per municipality
df_pop_full = (
    df_pop.set_index(["municipality_code", "year"])
    .groupby(level=0)
    .apply(lambda group: group.interpolate(method='linear', limit_direction='both'))
    .reset_index()
)

# -----------------------------
# 5. Expand to Daily Data
# -----------------------------
daily_dates = pd.date_range("2000-01-01", "2023-01-01", freq="D")
daily_index = pd.MultiIndex.from_product(
    [df_pop["municipality_code"].unique(), daily_dates],
    names=["municipality_code", "date"]
).to_frame(index=False)

df_pop_full["date"] = pd.to_datetime(df_pop_full["year"], format="%Y")
df_daily = daily_index.merge(df_pop_full[["municipality_code", "date", "POP"]], on=["municipality_code", "date"], how="left")

# Fill missing population values forward
df_daily["POP"] = (
    df_daily.sort_values(["municipality_code", "date"])
    .groupby("municipality_code")["POP"]
    .ffill()
)

# -----------------------------
# 6. Merge with Geographic Info
# -----------------------------
gdf_clean = gdf.drop(columns=["area_km2"])
df_out = df_daily.merge(gdf_clean, on="municipality_code", how="left")

# Add area back for density calculation
df_out = df_out.merge(gdf[["municipality_code", "area_km2"]], on="municipality_code", how="left")
df_out["pop_density"] = df_out["POP"] / df_out["area_km2"]

# -----------------------------
# 7. Compute Population Growth (% yearly change)
# -----------------------------
df_out = df_out.sort_values(["municipality_code", "date"])
df_out["pop_growth"] = (
    df_out.groupby("municipality_code")["POP"]
    .pct_change(periods=365) * 100
)
df_out["pop_growth"] = df_out["pop_growth"].ffill()

# -----------------------------
# 8. Final Column Order
# -----------------------------
final_cols = [
    "date", "municipality_code", "POP", "pop_density", "pop_growth",
    "province_code", "municipality_name", "lon", "lat", "geometry"
]
df_final = df_out[final_cols].sort_values(["municipality_code", "date"]).reset_index(drop=True)

# ✅ Done!
print(df_final.head())


## Extraccion Raster data de GHS BUILT


In [ ]:
# --- Load Level-2 shapefile: Provinces ---
gadm_lvl2 = gpd.read_file("/workspaces/JMT1ST-ECSF-External-Variables/src/gadm41_ESP_shp", layer="gadm41_ESP_2")
galicia_provinces = gadm_lvl2[gadm_lvl2["NAME_1"] == "Galicia"].to_crs(epsg=4326)

# --- Paths ---
folder_built = "/workspaces/JMT1ST-ECSF-External-Variables/data/GHS DATA/GHS-BUILT-S"
output_built_csv = "galicia_ghs_BUILT_by_province.csv"

# --- Init DF ---
province_names = galicia_provinces["NAME_2"].unique()
df_dict = {prov: {} for prov in province_names}

def extract_year(fname):
    match = re.search(r"(\d{4})", fname)
    return int(match.group(1)) if match else None

# --- Loop through BUILT rasters ---
print("🔧 Processing BUILT files by province...")
for fname in sorted(os.listdir(folder_built)):
    if not fname.endswith(".tif"):
        continue
    year = extract_year(fname)
    if year is None:
        print(f"[!] Skipped (no year): {fname}")
        continue

    path = os.path.join(folder_built, fname)

    try:
        with rasterio.open(path) as src:
            src_crs = src.crs
            provinces_reprojected = galicia_provinces.to_crs(src_crs)

            for _, row in provinces_reprojected.iterrows():
                prov_name = row["NAME_2"]
                geometry = [row["geometry"]]

                out_image, _ = mask(src, geometry, crop=True)
                data = out_image[0]
                if src.nodata is not None:
                    data = data[data != src.nodata]

                total = data.sum() if data.size > 0 else 0
                df_dict[prov_name][year] = total

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")

    gc.collect()

# --- Build full DataFrame ---
df_built = pd.DataFrame(df_dict)
df_built.index = pd.to_datetime([f"{y}-01-01" for y in df_built.index])
df_built = df_built.sort_index()

# --- Derive features ---
df_built_full = pd.DataFrame(index=df_built.index)

for prov in df_built.columns:
    prov_clean = prov.replace(' ', '_')
    col = f"built_{prov_clean}"
    df_built_full[col] = df_built[prov]

    # Growth: % change from previous year
    df_built_full[f"built_growth_{prov_clean}"] = df_built[prov].pct_change()

    # Built-up density = built / area (km²)
    area_km2 = galicia_provinces[galicia_provinces["NAME_2"] == prov].geometry.to_crs(epsg=3857).area.values[0] / 1e6
    df_built_full[f"built_density_{prov_clean}"] = df_built[prov] / area_km2

    # Share of total built-up in Galicia
    df_built_full[f"built_share_{prov_clean}"] = df_built[prov] / df_built.sum(axis=1)

# --- Done! ---
print("✅ BUILT data extracted and transformed by province.")
df_built_full.head()


## Extraccion Raster data de GHS SMOD

In [ ]:
# Load level-2 shapefile (provinces in Galicia)
gadm_lvl2 = gpd.read_file("/workspaces/JMT1ST-ECSF-External-Variables/src/gadm41_ESP_shp", layer="gadm41_ESP_2")
galicia_provinces = gadm_lvl2[gadm_lvl2["NAME_1"] == "Galicia"].to_crs(epsg=4326)

# Folder and output CSV
folder_smod = "/workspaces/JMT1ST-ECSF-External-Variables/data/GHS DATA/GHS-SMOD"
output_smod_csv = "galicia_ghs_SMOD_by_province.csv"

def extract_year(fname):
    match = re.search(r"(\d{4})", fname)
    return int(match.group(1)) if match else None

# Dictionary to collect results
province_names = galicia_provinces["NAME_2"].unique()
df_dict = {prov: {} for prov in province_names}

print("🔧 Processing SMOD files by province...")
for fname in sorted(os.listdir(folder_smod)):
    if not fname.endswith(".tif"):
        continue
    year = extract_year(fname)
    if year is None:
        print(f"[!] Skipped (no year): {fname}")
        continue

    path = os.path.join(folder_smod, fname)

    try:
        with rasterio.open(path) as src:
            src_crs = src.crs
            # Reproject provinces geometries to raster CRS once per file
            provinces_reprojected = galicia_provinces.to_crs(src_crs)

            for _, row in provinces_reprojected.iterrows():
                prov_name = row["NAME_2"]
                geometry = [row["geometry"]]  # List of shapely geometries for mask
                
                out_image, _ = mask(src, geometry, crop=True)
                data = out_image[0]
                if src.nodata is not None:
                    data = data[data != src.nodata]

                total = data.sum() if data.size > 0 else 0
                df_dict[prov_name][year] = total

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")

    gc.collect()

# Build DataFrame from dict
df_smod = pd.DataFrame(df_dict)
df_smod.index = pd.to_datetime([f"{y}-01-01" for y in df_smod.index])
df_smod = df_smod.sort_index()

# Add derived columns
df_smod_full = pd.DataFrame(index=df_smod.index)

for prov in df_smod.columns:
    prov_clean = prov.replace(' ', '_')
    col = f"smod_{prov_clean}"
    df_smod_full[col] = df_smod[prov]

    # Growth (% change year-to-year)
    df_smod_full[f"smod_growth_{prov_clean}"] = df_smod[prov].pct_change()

    # Density (sum / area in km²)
    area_km2 = galicia_provinces[galicia_provinces["NAME_2"] == prov].geometry.to_crs(epsg=3857).area.values[0] / 1e6
    df_smod_full[f"smod_density_{prov_clean}"] = df_smod[prov] / area_km2

    # Share of total SMOD
    df_smod_full[f"smod_share_{prov_clean}"] = df_smod[prov] / df_smod.sum(axis=1)

print("✅ SMOD data extracted and transformed by province.")
df_smod_full.head()
